# 03 · Fedlex

Swiss domestic federal law in German, French and Italian (SwissLawTranslations, article level), a held-out corpus. The code is `code/fedlex.py`. Results: `results/fedlex/raw/`.

1. **Cohort.** The three parquet files (`data/raw/fedlex/`; `train.parquet` is downloaded at the pinned revision on first use) →
   domestic federal law (SR 1–9 without international law and the 26 cantonal constitutions) → complete DE/FR/IT
   triples → triple-level exact-duplicate removal → a seeded sample of 2,784 triples (8,352 documents), checked
   against `data/fedlex_domestic_n2784_manifest.csv.gz`.
2. **Primary run** (Tables 2 and 3, the extended in-distribution and erasure-extent tables): Raw, PCA-1, PCA-2, k-means LEACE, Metadata-LEACE and SAE-LEACE on seeds
   0–4, each outer split holding out ≈50% of the articles with every law on one side.
3. **Ten-encoder study** (the ten-encoder table): the cells of `02_main_experiments.ipynb` §5 on Fedlex, seeds 0–2.

Each seed is checkpointed under `raw/*/checkpoints/` with a hash of its configuration (cohort, texts, embeddings,
recipe, conditions, split rule, audit); a checkpoint is reused only when the hash matches. Roughly an hour of GPU
time for the primary run and 15 hours for the ten encoders.

In [ ]:
# On Colab: mount Google Drive and move to your copy of the repository (see the README: "Running on Google Colab";
# set REPO if your copy is elsewhere). Skipped outside Colab.
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/label-free-deconfounding-embeddings'
    %cd {REPO}
    %pip install -q -r requirements.txt
except ImportError:
    print('not on Colab: using the current working directory')

In [ ]:
import os, sys
if not os.path.isdir('code') and os.path.isdir(os.path.join('..', 'code')):
    os.chdir('..')                                   # opened from notebooks/
assert os.path.isdir('code') and os.path.isdir('data'), f'run from the repository root; cwd={os.getcwd()}'
sys.path.insert(0, 'code')

import torch                                         # first: some modules load diptest, whose macOS wheel bundles libomp
import paths

# QUICK = True runs every section end to end in minutes on a subset (two seeds, a 20-epoch SAE, a few settings),
# writing to results/*_quick/ so it never mixes with the real results. Its numbers are NOT the paper's.
QUICK = False
paths.configure(quick_run=QUICK)                     # before the experiment modules are imported
DEVICE = 'cuda' if torch.cuda.is_available() else None   # None: each step's own choice
print('cwd:', os.getcwd(), '| device:', DEVICE or 'auto', '| QUICK' if QUICK else '')

In [ ]:
import artifacts, experiments as E
import fedlex as FL

artifacts.set_results_dir(paths.FEDLEX)
E.set_embed_cache(paths.FEDLEX_CACHE)              # dedicated to Fedlex; files named by corpus
RUN_SEEDS, NONLINEAR, ENC_SEEDS, ENCODERS, OVERRIDES = FL.RUN_SEEDS, FL.NONLINEAR, FL.ENC_SEEDS, FL.ENCODER_STUDY, {}
if QUICK:            # a 1,500-triple subsample of the cohort, one seed, E5-small only
    RUN_SEEDS, NONLINEAR, ENC_SEEDS, ENCODERS, OVERRIDES = (0,), (), (0,), ('e5-small',), dict(n_epochs=20)
FIT_DEVICE = DEVICE

## 1. The cohort

In [ ]:
CTX = FL.prepare(quick=QUICK)
DATASET = FL.save_dataset(CTX)
print(f"{CTX['n']:,} triples, {len(CTX['texts']):,} documents from {CTX['stats']['laws']:,} laws")
display(CTX['ledger'])

## 2. The primary run

In [ ]:
PRIMARY = FL.run_primary(CTX, seeds=RUN_SEEDS, nonlinear=NONLINEAR, device=DEVICE, fit_device=FIT_DEVICE, **OVERRIDES)
for c in FL.PRIMARY:
    a = PRIMARY['agg'][c]
    print(f"{FL.LABELS[c]:18s} linear AUC {a['linear_auc']['mean']:.3f} ± {a['linear_auc']['std']:.3f}"
          f"   R@10 {a['recall@10']['mean']:.3f} ± {a['recall@10']['std']:.3f}")

## 3. The ten-encoder study

In [ ]:
import gc
CELLS = {}
for enc in ENCODERS:
    CELLS[enc] = FL.encoder_cell(CTX, enc, seeds=ENC_SEEDS, primary=PRIMARY, device=DEVICE, fit_device=FIT_DEVICE,
                                 **OVERRIDES)
    gc.collect()
print(f"{len(CELLS)} encoder cells")